In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rauffauzanrambe/smart-city-traffic-flow-prediction-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/zadyra/.cache/kagglehub/datasets/rauffauzanrambe/smart-city-traffic-flow-prediction-dataset/versions/1


In [3]:
import os
import pandas as pd
print(os.listdir(path))

['road_network_international.csv', 'traffic_prediction_config.json', 'traffic_events.csv', 'road_network.csv', 'sensor_locations.json', 'traffic_sensor_data.csv', 'weather_conditions.csv']


# Dataset File Guide

| File | Rows | What it contains |
|---|---|---|
| `traffic_sensor_data.csv` | 218,400 | Core dataset — 30-min readings from road sensors (Jan–Mar 2024): vehicle counts by type, speed, occupancy, congestion level |
| `weather_conditions.csv` | 21,840 | Hourly weather per station: temperature, rain, wind, visibility, air quality, UV |
| `traffic_events.csv` | 2,000 | Traffic incidents: accidents, breakdowns, fallen trees — with severity, duration, response time, casualties |
| `road_network.csv` | 200 | Jakarta local roads: type, length, lanes, speed limit, capacity, surface condition, district |
| `road_network_international.csv` | 1,401 | International road comparison (Tokyo + other cities): 41 features including CO2, noise, ITS, signal types |
| `sensor_locations.json` | 10 sensors | GPS coordinates, location names, and directions covered by each sensor |
| `traffic_prediction_config.json` | — | ML system config: LSTM model settings, feature engineering rules, deployment info |


# Load All Files

In [4]:
import pandas as pd
import json, os

DATA = path  # from kagglehub download above

sensors  = pd.read_csv(os.path.join(DATA, 'traffic_sensor_data.csv'), parse_dates=['timestamp'])
weather  = pd.read_csv(os.path.join(DATA, 'weather_conditions.csv'),  parse_dates=['timestamp'])
events   = pd.read_csv(os.path.join(DATA, 'traffic_events.csv'),      parse_dates=['timestamp_start','timestamp_end'])
roads    = pd.read_csv(os.path.join(DATA, 'road_network.csv'))
roads_int= pd.read_csv(os.path.join(DATA, 'road_network_international.csv'))

with open(os.path.join(DATA, 'sensor_locations.json')) as f:
    sensor_meta = json.load(f)

print(f'traffic_sensor_data : {sensors.shape}')
print(f'weather_conditions  : {weather.shape}')
print(f'traffic_events      : {events.shape}')
print(f'road_network        : {roads.shape}')
print(f'road_network_intl   : {roads_int.shape}')
print(f'sensors in JSON     : {len(sensor_meta["sensors"])}')


traffic_sensor_data : (218400, 15)
weather_conditions  : (21840, 11)
traffic_events      : (2000, 15)
road_network        : (200, 18)
road_network_intl   : (1401, 41)
sensors in JSON     : 50


---
# 1 — Peak Hours & Congestion Patterns

In [5]:
print('--- Average vehicle count by hour (all sensors) ---')
hourly = sensors.groupby('hour')['vehicle_count'].mean().round(1)
print(hourly.to_string())
print(f'\nPeak hour:    {hourly.idxmax()}:00  ({hourly.max():.1f} vehicles avg)')
print(f'Quietest hour:{hourly.idxmin()}:00  ({hourly.min():.1f} vehicles avg)')

print('\n--- Congestion level distribution ---')
cong_dist = sensors['congestion_level'].value_counts(normalize=True).mul(100).round(1)
print(cong_dist.to_string())

print('\n--- Congestion by hour (% High or Critical) ---')
sensors['is_high_cong'] = sensors['congestion_level'].isin(['High','Critical'])
cong_hour = sensors.groupby('hour')['is_high_cong'].mean().mul(100).round(1)
print(cong_hour.sort_values(ascending=False).head(10).to_string())

print('\n--- Congestion by day of week (0=Mon ... 6=Sun) ---')
day_names = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}
cong_day = sensors.groupby('day_of_week')[['vehicle_count','average_speed_kmh','occupancy_rate']].mean().round(2)
cong_day.index = cong_day.index.map(day_names)
print(cong_day.to_string())


--- Average vehicle count by hour (all sensors) ---
hour
0     15.4
1     15.4
2     15.4
3     15.3
4     15.2
5     15.2
6     15.3
7     62.4
8     62.2
9     62.4
10    45.2
11    45.3
12    45.1
13    45.0
14    45.3
15    44.3
16    44.5
17    75.5
18    75.7
19    75.8
20    36.5
21    27.0
22    27.2
23    15.3

Peak hour:    19:00  (75.8 vehicles avg)
Quietest hour:4:00  (15.2 vehicles avg)

--- Congestion level distribution ---
congestion_level
Low       71.2
Medium    28.8

--- Congestion by hour (% High or Critical) ---
hour
0     0.0
1     0.0
22    0.0
21    0.0
20    0.0
19    0.0
18    0.0
17    0.0
16    0.0
15    0.0

--- Congestion by day of week (0=Mon ... 6=Sun) ---
             vehicle_count  average_speed_kmh  occupancy_rate
day_of_week                                                  
Mon                  43.26              55.36            0.26
Tue                  43.21              55.46            0.26
Wed                  43.39              55.57           

---
# 2 — Worst & Best Sensor Locations

In [6]:
loc_stats = sensors.groupby('location').agg(
    avg_vehicles    =('vehicle_count',    'mean'),
    avg_speed_kmh   =('average_speed_kmh','mean'),
    avg_occupancy   =('occupancy_rate',   'mean'),
    pct_high_cong   =('is_high_cong',     'mean'),
).round(3)
loc_stats['pct_high_cong'] = (loc_stats['pct_high_cong']*100).round(1)
loc_stats = loc_stats.sort_values('pct_high_cong', ascending=False)

print('--- Locations ranked by % time in High/Critical congestion ---')
print(loc_stats.to_string())

print('\n--- Slowest locations (lowest avg speed) ---')
print(loc_stats.sort_values('avg_speed_kmh').head(5)[['avg_speed_kmh','avg_vehicles']].to_string())

print('\n--- Busiest locations (most vehicles) ---')
print(loc_stats.sort_values('avg_vehicles', ascending=False).head(5)[['avg_vehicles','avg_speed_kmh']].to_string())


--- Locations ranked by % time in High/Critical congestion ---
                                        avg_vehicles  avg_speed_kmh  avg_occupancy  pct_high_cong
location                                                                                         
Jl. Alternatif Cibubur - Depok                39.530         57.967          0.241            0.0
Jl. Raya Kebayoran - Blok M                   39.521         57.838          0.241            0.0
Jl. Perintis Kemerdekaan - Cakung             39.558         57.949          0.242            0.0
Jl. Pluit - Muara Karang                      39.035         57.969          0.241            0.0
Jl. Pondok Indah - Pondok Pinang              39.465         58.186          0.242            0.0
Jl. Proklamasi - Menteng                      39.085         58.041          0.241            0.0
Jl. Raden Saleh - Salemba                     39.121         57.889          0.240            0.0
Jl. Rasuna Said - Setiabudi                   38.945   

---
# 3 — Vehicle Mix & Holiday Effect

In [7]:
print('--- Overall vehicle mix (% of total) ---')
mix_cols = ['car_count','motorcycle_count','truck_count','bus_count']
totals = sensors[mix_cols].sum()
pct = (totals / totals.sum() * 100).round(1)
print(pct.to_string())

print('\n--- Vehicle mix on HOLIDAYS vs normal days ---')
for grp, label in [(1,'Holiday'),(0,'Normal')]:
    sub = sensors[sensors['is_holiday']==grp]
    t = sub[mix_cols].sum()
    p = (t / t.sum() * 100).round(1)
    print(f'  {label}: {dict(zip(["Cars","Motos","Trucks","Buses"], p.values))}')

print('\n--- Holiday vs normal day key metrics ---')
hol = sensors.groupby('is_holiday')[['vehicle_count','average_speed_kmh','occupancy_rate']].mean().round(2)
hol.index = hol.index.map({0:'Normal day',1:'Holiday'})
print(hol.to_string())

print('\n--- Morning rush (7-9am) vs Evening rush (5-7pm) vs Off-peak ---')
def period(h):
    if 7<=h<=9:  return 'Morning rush'
    if 17<=h<=19: return 'Evening rush'
    return 'Off-peak'
sensors['period'] = sensors['hour'].apply(period)
rush = sensors.groupby('period')[['vehicle_count','average_speed_kmh','occupancy_rate','is_high_cong']].mean().round(3)
rush['is_high_cong'] = (rush['is_high_cong']*100).round(1)
print(rush.rename(columns={'is_high_cong':'pct_high_cong%'}).to_string())


--- Overall vehicle mix (% of total) ---
car_count           41.2
motorcycle_count    36.2
truck_count          5.2
bus_count           17.3

--- Vehicle mix on HOLIDAYS vs normal days ---
  Holiday: {'Cars': np.float64(40.9), 'Motos': np.float64(35.9), 'Trucks': np.float64(4.9), 'Buses': np.float64(18.3)}
  Normal: {'Cars': np.float64(41.4), 'Motos': np.float64(36.3), 'Trucks': np.float64(5.3), 'Buses': np.float64(17.0)}

--- Holiday vs normal day key metrics ---
            vehicle_count  average_speed_kmh  occupancy_rate
is_holiday                                                  
Normal day          43.32              55.43            0.26
Holiday             30.94              63.17            0.20

--- Morning rush (7-9am) vs Evening rush (5-7pm) vs Off-peak ---
              vehicle_count  average_speed_kmh  occupancy_rate  pct_high_cong%
period                                                                        
Evening rush         75.674             31.298           0.441 

---
# 4 — Weather Impact on Traffic
Merge sensor readings with weather station data on the nearest timestamp.

In [8]:
# Round both to hour for merge
sensors['ts_hour'] = sensors['timestamp'].dt.floor('h')
weather['ts_hour'] = weather['timestamp'].dt.floor('h')
weather_avg = weather.groupby('ts_hour')[['temperature_c','precipitation_mm','visibility_km','air_quality_index','wind_speed_kmh']].mean()

merged = sensors.merge(weather_avg, on='ts_hour', how='left')

print('--- Traffic by weather type ---')
# Re-merge with weather_type (categorical)
wtype = weather.groupby('ts_hour')['weather_type'].agg(lambda x: x.mode()[0]).reset_index()
merged2 = merged.merge(wtype, on='ts_hour', how='left')
wt_stats = merged2.groupby('weather_type')[['vehicle_count','average_speed_kmh','occupancy_rate','is_high_cong']].mean().round(3)
wt_stats['is_high_cong'] = (wt_stats['is_high_cong']*100).round(1)
print(wt_stats.rename(columns={'is_high_cong':'pct_high_cong%'}).sort_values('pct_high_cong%', ascending=False).to_string())

print('\n--- Correlation: weather variables vs traffic ---')
weather_cols = ['precipitation_mm','visibility_km','temperature_c','wind_speed_kmh','air_quality_index']
traffic_cols = ['vehicle_count','average_speed_kmh','occupancy_rate']
corr = merged[weather_cols + traffic_cols].corr().loc[weather_cols, traffic_cols].round(3)
print(corr.to_string())

print('\n--- Heavy rain effect: rain >5mm vs dry ---')
merged['rain_heavy'] = merged['precipitation_mm'] > 5
rain_comp = merged.groupby('rain_heavy')[['vehicle_count','average_speed_kmh','is_high_cong']].mean().round(3)
rain_comp['is_high_cong'] = (rain_comp['is_high_cong']*100).round(1)
rain_comp.index = rain_comp.index.map({False:'Dry / light rain',True:'Heavy rain (>5mm)'})
print(rain_comp.to_string())


--- Traffic by weather type ---
               vehicle_count  average_speed_kmh  occupancy_rate  pct_high_cong%
weather_type                                                                   
Badai Petir           37.331             59.767           0.231             0.0
Berawan               39.758             57.722           0.244             0.0
Cerah Berawan         38.648             58.463           0.238             0.0
Hujan Lebat           40.197             57.351           0.246             0.0
Hujan Ringan          39.306             57.949           0.242             0.0
Hujan Sedang          39.983             56.972           0.245             0.0
Kabut                 38.356             58.926           0.235             0.0

--- Correlation: weather variables vs traffic ---
                   vehicle_count  average_speed_kmh  occupancy_rate
precipitation_mm          -0.000             -0.000          -0.001
visibility_km              0.001              0.001          

---
# 5 — Traffic Events (Incidents) Analysis

In [9]:
print('--- Event types (translated) ---')
# Indonesian → English mapping
type_map = {
    'Kendaraan Mogok':'Broken-down vehicle',
    'Pohon Tumbang':'Fallen tree',
    'Kecelakaan':'Accident',
    'Banjir':'Flooding',
    'Perbaikan Jalan':'Road works',
    'Demonstrasi':'Demonstration/Protest',
}
events['event_type_en'] = events['event_type'].map(type_map).fillna(events['event_type'])
print(events['event_type_en'].value_counts().to_string())

print('\n--- Severity distribution ---')
print(events['severity'].value_counts().to_string())

print('\n--- Avg duration & response time by event type ---')
ev_stats = events.groupby('event_type_en').agg(
    count           =('event_id',          'count'),
    avg_duration_min=('duration_minutes',  'mean'),
    avg_response_min=('response_time_min', 'mean'),
    avg_lanes_hit   =('affected_lanes',    'mean'),
    total_casualties=('casualties',        'sum'),
).round(1).sort_values('avg_duration_min', ascending=False)
print(ev_stats.to_string())

print('\n--- Events by hour of day ---')
events['hour'] = events['timestamp_start'].dt.hour
print(events.groupby('hour')['event_id'].count().sort_values(ascending=False).head(8).to_string())

print('\n--- Events by weather condition at time of incident ---')
print(events['weather_condition'].value_counts().head(8).to_string())

print('\n--- Most incident-prone roads ---')
print(events['location_description'].value_counts().head(8).to_string())

print('\n--- Response team dispatched rate by severity ---')
resp = events.groupby('severity')['response_team_dispatched'].mean().mul(100).round(1)
print(resp.to_string())


--- Event types (translated) ---
event_type_en
Kemacetan Parah        212
Fallen tree            210
Broken-down vehicle    204
Protes/Unjuk Rasa      202
Jalan Ditutup          200
Konvoi                 199
Acara Khusus           198
Accident               194
Flooding               191
Road works             190

--- Severity distribution ---
severity
Moderate    753
Major       589
Minor       443
Critical    215

--- Avg duration & response time by event type ---
                     count  avg_duration_min  avg_response_min  avg_lanes_hit  total_casualties
event_type_en                                                                                  
Road works             190            2332.9              17.5            2.5                 0
Acara Khusus           198             439.1              20.4            2.5                 0
Flooding               191             321.4              20.8            2.5                 0
Jalan Ditutup          200             203.2   

---
# 6 — Jakarta Road Network Analysis

In [10]:
print('--- Road types distribution ---')
print(roads['road_type'].value_counts().to_string())

print('\n--- Surface condition distribution ---')
print(roads['surface_condition'].value_counts().to_string())

print('\n--- Capacity utilization by district (avg_daily_traffic / capacity) ---')
roads['utilization_pct'] = (roads['avg_daily_traffic'] / roads['capacity_vehicles_per_hour'] * 100).round(1)
dist_util = roads.groupby('district').agg(
    roads_count     =('road_id',          'count'),
    avg_utilization =('utilization_pct',  'mean'),
    avg_daily_traffic=('avg_daily_traffic','mean'),
    avg_capacity    =('capacity_vehicles_per_hour','mean'),
).round(1).sort_values('avg_utilization', ascending=False)
print(dist_util.to_string())

print('\n--- Surface condition vs avg daily traffic ---')
print(roads.groupby('surface_condition')['avg_daily_traffic'].mean().round(0).sort_values(ascending=False).to_string())

print('\n--- Roads overdue for maintenance (last_maintenance_date before 2023) ---')
roads['last_maint'] = pd.to_datetime(roads['last_maintenance_date'])
overdue = roads[roads['last_maint'] < '2023-01-01'][['road_name','district','surface_condition','last_maintenance_date','avg_daily_traffic']]
print(f'Overdue roads: {len(overdue)} out of {len(roads)}')
print(overdue.sort_values('avg_daily_traffic', ascending=False).head(10).to_string(index=False))


--- Road types distribution ---
road_type
Jalan Tol           48
Jalan Lingkungan    46
Jalan Kolektor      44
Jalan Lokal         33
Jalan Arteri        29

--- Surface condition distribution ---
surface_condition
Baik            106
Cukup            61
Rusak Ringan     23
Rusak Berat      10

--- Capacity utilization by district (avg_daily_traffic / capacity) ---
               roads_count  avg_utilization  avg_daily_traffic  avg_capacity
district                                                                    
Cawang                  10            139.4             4647.9        3682.3
Cibubur                 10            134.3             8047.9        6306.5
Sudirman                12            134.0             5430.3        4744.2
Menteng                 16            133.3             6023.1        4330.7
Slipi                    5            131.7            10560.8        7725.4
Kemang                   5            128.5             7181.8        5926.6
Bekasi          

---
# 7 — International Road Network Comparison

In [11]:
print('--- Countries & cities in dataset ---')
print(roads_int.groupby('country')['city'].nunique().sort_values(ascending=False).to_string())

print('\n--- Smart infrastructure adoption by country (% of roads with ITS) ---')
smart = roads_int.groupby('country').agg(
    pct_ITS         =('has_intelligent_transport_system','mean'),
    pct_speed_camera=('has_speed_camera',               'mean'),
    pct_bus_lane    =('has_bus_lane',                   'mean'),
    pct_bike_lane   =('has_bike_lane',                  'mean'),
    avg_noise_db    =('noise_level_db',                 'mean'),
    avg_co2_tons    =('annual_co2_emission_tons',       'mean'),
).mul({'pct_ITS':100,'pct_speed_camera':100,'pct_bus_lane':100,'pct_bike_lane':100,'avg_noise_db':1,'avg_co2_tons':1}).round(1)
print(smart.sort_values('pct_ITS', ascending=False).to_string())

print('\n--- CO2 emissions by road type (all countries) ---')
print(roads_int.groupby('road_type')['annual_co2_emission_tons'].mean().round(1).sort_values(ascending=False).to_string())

print('\n--- Avg daily traffic by country ---')
print(roads_int.groupby('country')['avg_daily_traffic'].mean().round(0).sort_values(ascending=False).to_string())

print('\n--- Speed limits across countries (avg) ---')
print(roads_int.groupby('country')['speed_limit_kmh'].mean().round(1).sort_values(ascending=False).to_string())

print('\n--- Congestion charge usage by country ---')
print(roads_int.groupby('country')['has_congestion_charge'].mean().mul(100).round(1).sort_values(ascending=False).head(10).to_string())


--- Countries & cities in dataset ---
country
Australia         1
Brazil            1
United Kingdom    1
UAE               1
Thailand          1
Sweden            1
South Korea       1
Singapore         1
Russia            1
Nigeria           1
Netherlands       1
Mexico            1
Japan             1
India             1
Germany           1
France            1
Egypt             1
China             1
Canada            1
United States     1

--- Smart infrastructure adoption by country (% of roads with ITS) ---
                pct_ITS  pct_speed_camera  pct_bus_lane  pct_bike_lane  avg_noise_db  avg_co2_tons
country                                                                                           
Brazil             56.7              40.0          16.7           23.3          69.3         209.8
Australia          56.3              42.3          32.4            8.5          71.1         205.7
Mexico             55.7              40.0          20.0           15.7          69.1  

---
# 8 — Cross-File: Are Events Worse During Bad Weather?

In [12]:
# Tag event hours with weather
events['ts_hour'] = events['timestamp_start'].dt.floor('h')
weather_avg2 = weather.groupby('ts_hour')[['precipitation_mm','visibility_km','temperature_c']].mean().reset_index()
ev_wx = events.merge(weather_avg2, on='ts_hour', how='left')

print('--- Avg precipitation during each event type ---')
print(ev_wx.groupby('event_type_en')['precipitation_mm'].mean().round(2).sort_values(ascending=False).to_string())

print('\n--- Visibility during Major vs Minor events ---')
print(ev_wx.groupby('severity')['visibility_km'].mean().round(2).to_string())

print('\n--- Avg sensor speed during event hours vs non-event hours ---')
event_hours = set(events['ts_hour'].dt.to_pydatetime())
sensors['in_event_hour'] = sensors['ts_hour'].isin(event_hours)
print(sensors.groupby('in_event_hour')[['vehicle_count','average_speed_kmh','occupancy_rate','is_high_cong']].mean().round(3).rename(index={False:'No event',True:'Event hour'}).to_string())

print('\n--- Which hour of day do events cause the most slowdown? ---')
ev_hours = sensors[sensors['in_event_hour']].groupby('hour')['average_speed_kmh'].mean().round(1)
no_ev_hours = sensors[~sensors['in_event_hour']].groupby('hour')['average_speed_kmh'].mean().round(1)
impact = (no_ev_hours - ev_hours).dropna().sort_values(ascending=False)
print('Speed reduction (km/h) caused by events, by hour:')
print(impact.head(8).to_string())


--- Avg precipitation during each event type ---
event_type_en
Acara Khusus           9.81
Flooding               9.46
Kemacetan Parah        9.27
Jalan Ditutup          8.85
Road works             8.71
Fallen tree            8.44
Konvoi                 8.35
Accident               8.33
Broken-down vehicle    8.13
Protes/Unjuk Rasa      7.19

--- Visibility during Major vs Minor events ---
severity
Critical    5.05
Major       5.51
Minor       5.84
Moderate    5.52

--- Avg sensor speed during event hours vs non-event hours ---
               vehicle_count  average_speed_kmh  occupancy_rate  is_high_cong
in_event_hour                                                                
No event              39.006             58.160           0.240           0.0
Event hour            39.385             57.864           0.242           0.0

--- Which hour of day do events cause the most slowdown? ---
Speed reduction (km/h) caused by events, by hour:
hour
7     6.5
17    2.5
21    1.6
14    1.

/var/folders/2x/kps_pkdd2r97_klg02p4z6xc0000gn/T/ipykernel_24028/2437459686.py:13: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  event_hours = set(events['ts_hour'].dt.to_pydatetime())
